# Final Corpus EDA

Use this notebook for the thesis-facing EDA based on the final cleaned corpus.

Primary outputs for the thesis:
- corpus size by outlet
- corpus size by month
- publication timeline
- article-length distributions
- normalized comparisons for Section 3.3

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path.cwd()
OUTPUT_FIGURES = BASE / 'outputs' / 'figures'
OUTPUT_TABLES = BASE / 'outputs' / 'tables'

CLEANED_FILES = {
    'rt_de': Path('../rt_de_clean.csv'),
    'compact': Path('../compact_clean.csv'),
    'nius': Path('../nius_clean.csv'),
    'tichys': Path('../tichys_clean.csv'),
    'antispiegel': Path('../antispiegel_clean.csv'),
    'tagesschau': Path('../tagesschau_clean.csv'),
    'dkurier': Path('../dkurier_clean.csv'),
}

for path in [OUTPUT_FIGURES, OUTPUT_TABLES]:
    path.mkdir(parents=True, exist_ok=True)

## 1. Load and Combine Final Cleaned Corpora

Goal: build the exact corpus that will be described in the thesis.

In [ ]:
OUTLET_LABELS = {
    'rt_de': 'RT DE',
    'compact': 'Compact',
    'nius': 'NIUS',
    'tichys': 'Tichys Einblick',
    'antispiegel': 'Anti-Spiegel',
    'tagesschau': 'Tagesschau',
    'dkurier': 'Deutschland-Kurier',
}

frames = []

for outlet_key, path in CLEANED_FILES.items():
    df = pd.read_csv(path).copy()
    df['Title'] = df['Title'].fillna('').str.strip()
    df['Text'] = df['Text'].fillna('').str.strip()
    df['source'] = df['source'].fillna(OUTLET_LABELS[outlet_key]).str.strip()
    df['outlet_key'] = outlet_key
    df['outlet'] = OUTLET_LABELS[outlet_key]
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True)
    df['date'] = df['Date'].dt.date
    df['year_month'] = df['Date'].dt.tz_convert(None).dt.to_period('M').astype(str)
    df['text_length_chars'] = df['Text'].str.len()
    df['text_length_words'] = df['Text'].str.split().str.len()
    frames.append(df)

df_final_corpus = pd.concat(frames, ignore_index=True)
df_final_corpus = df_final_corpus.sort_values(['Date', 'outlet']).reset_index(drop=True)

corpus_overview = pd.DataFrame({
    'rows': [len(df_final_corpus)],
    'outlets': [df_final_corpus['outlet'].nunique()],
    'date_min': [df_final_corpus['Date'].min()],
    'date_max': [df_final_corpus['Date'].max()],
})

display(corpus_overview)
display(df_final_corpus[['Date', 'outlet', 'Title', 'source']].head())

## 2. Corpus Size by Outlet and Month

Suggested exports:
- one summary table for all outlets
- one monthly count table or long-format export

## 3. Publication Timeline

Suggested figure:
- monthly publication counts by outlet
- or a faceted time-series plot

## 4. Article Length Distributions

Suggested outputs:
- descriptive summary table
- histogram or boxplot by outlet

## 5. Notes for Section 3.3

Record short takeaways that can be translated into the thesis text:
- which outlets are largest or smallest
- whether some months are unusually dense
- whether article lengths differ substantially
- why normalized comparisons are needed